In [ ]:
# !pip install langchain
# !pip install langchain-community
# !pip install pypdf

In [ ]:
# !pip install python-dotenv huggingface_hub

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# import os
# from dotenv import load_dotenv

# load_dotenv()

# hf_token = os.getenv("HF_TOKEN")
# print(hf_token)  # debug once, then remove

In [ ]:
# csv_data = "https://raw.githubusercontent.com/Shikher-jain/LLM-Course/refs/heads/main/DataCamp/csv_data.csv"
# html_data = "https://raw.githubusercontent.com/Shikher-jain/LLM-Course/refs/heads/main/DataCamp/html_data.html"
# pdf_data = "https://raw.githubusercontent.com/Shikher-jain/LLM-Course/refs/heads/main/DataCamp/pdf_data.pdf"

In [ ]:
import pprint

In [ ]:
# Import library
from langchain_community.document_loaders import PyPDFLoader, UnstructuredHTMLLoader

# Create a document loader for rag_paper.pdf
loader = PyPDFLoader('pdf_data.pdf')

# Load the document
pdf_data = loader.load()

print(pdf_data[0].page_content)
print(pdf_data[0].metadata)

In [ ]:
# !pip install unstructured

In [ ]:
# Import library
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Load the document
loader= UnstructuredHTMLLoader("html_data.html")
html_data = loader.load()

# Print the first document's content
print(html_data[0].page_content)

# Print the first document's metadata
print(html_data[0].metadata)

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("csv_data.csv")
csv_data = loader.load()

print(csv_data[0].page_content)
print(csv_data[-1].metadata)

In [ ]:
# !pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator='.',
    chunk_size=75,
    chunk_overlap=10
)
# Split the text using text_splitter
chunks = text_splitter.split_text(csv_data[0].page_content)
pprint.pprint(chunks)
pprint.pprint([len(chunk) for chunk in chunks])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=['.'],
    chunk_size=75,
    chunk_overlap=10
)

# Split the text using text_splitter
# chunks = text_splitter.split_text(csv_data[-1].page_content)
chunks = text_splitter.split_documents(pdf_data)
pprint.pprint(chunks)
# print([len(chunk[0]) for chunk in chunks.page_content])
# print([len(chunk[0]) for chunk.page_content in chunks])
print([len(chunk.page_content) for chunk in chunks])

In [ ]:
# !pip install langchain_chroma langchain_openai
# !pip install langchain_community
# !pip install sentence-transformers
# !pip install -U chromadb langchain langchain-community langchain-core
# !pip install -U opentelemetry-api opentelemetry-sdk
# !pip install opentelemetry-sdk==1.19.0 opentelemetry-api==1.19.0


In [ ]:
# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Create an instance of the OpenAIEmbeddings class

# embedding_model = OpenAIEmbeddings(
    # api_key="<OPENAI_API_TOKEN>",
    #   model='text-embedding-3-small')

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)


In [ ]:
# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Create an instance of the OpenAIEmbeddings class

# embedding_model = OpenAIEmbeddings(
    # api_key="<OPENAI_API_TOKEN>",
    #   model='text-embedding-3-small')

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
    search_type="similarity",
    # search_type="mmr",
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# prompt = ChatPromptTemplate.from_messages()
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question based on the context provided.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
)

In [ ]:
# llm = ChatOpenAI(model="gpt-3.5-turbo",api_key="your-api-key", temperature=0)

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# pipe = pipeline(
#     "text-generation",
#     model="mistralai/Mistral-7B-Instruct-v0.2",
#     device=-0  # GPU (use -1 for CPU)
# )
pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    device=-1
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# !pip install ipywidgets

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
result = chain.invoke("What is the main topic of the document?")
print(result)

In [ ]:

prompt = """
Use the only the context provided to answer the following question. If you don't know the answer, reply that you are unsure.
Context: {context}
Question: {question}
"""

from langchain_core.prompts import ChatPromptTemplate


# Convert the string into a chat prompt template
prompt_template = ChatPromptTemplate.from_template(prompt)

# Create an LCEL chain to test the prompt
chain = prompt_template | llm

# Invoke the chain on the inputs provided
print(chain.invoke({"context": "DataCamp's RAG course was created by Meri Nova and James Chapman!", "question": "Who created DataCamp's RAG course?"}))

In [ ]:
# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":2})

# Create the LCEL retrieval chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# Invoke the chain
print(chain.invoke("Who are the authors?"))

In [ ]:
# !pip install markdown

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

loader = UnstructuredMarkdownLoader("markdown_data.md")
markdown_data = loader.load()
print(markdown_data[0].page_content)

In [ ]:
from langchain_community.document_loaders import PythonLoader

loader = PythonLoader("python_data.py")
python_data = loader.load()

print(python_data[0].page_content)

In [ ]:
python_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=10)

chunks = python_splitter.split_documents(python_data)

In [ ]:
pprint.pprint(chunks)

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk.page_content)
    print("\n---\n")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=100,
    chunk_overlap=10)

chunks = python_splitter.split_documents(python_data)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk.page_content)
    print("\n---\n")

In [ ]:
import tiktoken
from langchain_text_splitters import TokenTextSplitter

example_text = "This is an example text to demonstrate token splitting. It will be split into chunks based on the specified chunk size and overlap."
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
# encoding = tiktoken.encoding_for_model("cl100k_base")
# token_splitter = TokenTextSplitter(encoding_name=encoding, chunk_size=10, chunk_overlap=2)
token_splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=10, chunk_overlap=2)
chunks = token_splitter.split_text(example_text)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} \nNo. Tokens: {len(encoding.encode(chunk))}")
    print(chunk)
    print("\n---\n")

In [ ]:

# !pip install langchain-experimental

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

semantic_chunker = SemanticChunker(embeddings=embeddings, breakpoint_threshold_amount=0.8,breakpoint_threshold_type="gradient")

In [ ]:
chuks = semantic_chunker.split_documents(pdf_data)
for i, chunk in enumerate(chuks):    
    print(f"Chunk {i+1}:")
    print(chunk.page_content)
    print("\n---\n")

In [ ]:
# !pip install rank_bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever

chunks= [
    'Python is a high-level programming language known for its simplicity.',
    'It is widely used in data science, machine learning, and web development.',
    'Python has a large ecosystem of libraries like NumPy, pandas, and PyTorch.',
    'It supports multiple programming paradigms including object-oriented and functional programming.'
]

bm25_retriever = BM25Retriever.from_texts(chunks,k=3)
results = bm25_retriever.invoke("What are some libraries in Python?")
pprint.pprint(results)

In [ ]:
retriever = BM25Retriever.from_documents(csv_data,k=5)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

pprint.pprint(chain.invoke("What is the main topic of the document?"))
# retriever.get_relevant_documents("What is the main topic of the document?")


In [ ]:
query = "What is the main topic of the document?"
predicted_answer ="Information security, cybersecurity and privacy protection"
ref_answer = "Cybersecurity"

In [ ]:
# from langchain.prompts import PromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFaceHub

In [ ]:
prompt_template = """You are a helpful assistant. Evaluate the predicted answer to the question based on the reference answer. If the predicted answer is correct, reply with "Correct". If the predicted answer is incorrect, reply with "Incorrect".\n\nQuestion: {query}\nPredicted Answer: {predicted_answer}\nReference Answer: {ref_answer}\nEvaluation:"""

prompt = PromptTemplate(
    input_variables=["query", "predicted_answer", "ref_answer"],
    template=prompt_template
)
import os
from dotenv import load_dotenv

# Load .env
load_dotenv()

# Set HF token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HF_TOKEN")
eval_llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",   # 🔥 good balance
    model_kwargs={
        "temperature": 0,
        "max_new_tokens": 50
    }
)
# eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# eval_llm = ChatOpenAI(model="gpt-3.5-turbo",api_key="your-api-key/", temperature=0)

In [ ]:
# !pip install langchain_huggingface

In [ ]:
!pip install huggingface_hub==0.20.3

In [ ]:
# from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
# from langchain_core.messages import HumanMessage
# import os

# llm_base = HuggingFaceEndpoint(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.2",  # ✅ works
#     task="text-generation",
#     temperature=0,
#     max_new_tokens=50,
#     huggingfacehub_api_token=os.getenv("HF_TOKEN")
# )
# # Step 2: wrap it
# llm = ChatHuggingFace(llm=llm_base)

# # from transformers import pipeline
# # from langchain_community.llms import HuggingFacePipeline
# # pipe = pipeline("text2text-generation", model="google/flan-t5-base")

# # llm = HuggingFacePipeline(pipeline=pipe)

# # Step 3: use it
# def qa_evaluator(query, pred, ref):
#     msg = f"""
# Return ONLY 'Correct' or 'Incorrect'.

# Question: {query}
# Predicted Answer: {pred}
# Reference Answer: {ref}
# """

#     response = llm.invoke([
#         HumanMessage(content=msg)
#     ])

#     result = response.content.strip().lower()
#     return 1 if "correct" in result else 0

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

# ✅ ONLY ONE LLM (local)
# pipe = pipeline("text2text-generation", model="google/flan-t5-base")
pipe = pipeline("text-generation", model="google/flan-t5-base")
llm = HuggingFacePipeline(pipeline=pipe)

# Prompt
prompt = PromptTemplate(
    input_variables=["query", "predicted_answer", "ref_answer"],
    template="""
Return ONLY 'Correct' or 'Incorrect'.

Question: {query}
Predicted Answer: {predicted_answer}
Reference Answer: {ref_answer}
"""
)

# Chain
chain = prompt | llm

# Evaluator
def qa_evaluator(query, pred, ref):
    result = chain.invoke({
        "query": query,
        "predicted_answer": pred,
        "ref_answer": ref
    })

    result = result.strip().lower()
    print("RAW:", result)

    return 1 if "correct" in result else 0


# TEST
query = "What is the main topic of the document?"
predicted_answer = "Information security, cybersecurity and privacy protection"
ref_answer = "Cybersecurity"

score = qa_evaluator(query, predicted_answer, ref_answer)
print("Score:", score)

In [ ]:
# # !pip install ragas
# !pip install langchain==0.0.350 ragas==0.0.22

In [ ]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.llms import HuggingFaceEndpoint
from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness

# llm = ChatOpenAI(model="gpt-40-mini", api_key="...")

llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    temperature=0,
    max_new_tokens=256,
    huggingfacehub_api_token=os.getenv("HF_TOKEN")
)
# Embeddings (local)
embeddings = SentenceTransformer("all-MiniLM-L6-v2")

faithfulness_chain = EvaluatorChain(
    metric=faithfulness,
    llm=llm,
    embeddings=embeddings
)
# embeddings = OpenAIEmbeddings (model="text-embedding-3-small", api_key="...")
# faithfulness_chain = EvaluatorChain(
#     metric=faithfulness,
#     llm=llm,
#     embeddings=embeddings
# )

In [ ]:
eval_result = faithfulness_chain({
"question": "How does the RAG model improve question answering with LLMs?",
"answer": "The RAG model improves question answering by combining the retrieval of documents.",
"contexts": [
"The RAG model integrates document retrieval with LLMs by first retrieving relevant passages from a knowledge base and then using those passages as context for the LLM to generate an answer.",
"By incorporating retrieval mechanisms, RAG leverages external knowledge sources, allowing it to provide more accurate and informed responses.",
]
})
print(eval_result)

In [ ]:
from ragas.metrics import context_precision
llm = ChatOpenAI (model="gpt-40-mini", api_key="...")
embeddings = OpenAIEmbeddings (model="text-embedding-3-small", api_key="...")
context_precision_chain = EvaluatorChain(
metric=context_precision,
LLm=LLm,
embeddings=embeddings
)

In [ ]:
eval_result = context_precision_chain({
"question": "How does the RAG model improve question answering with large language models?",
"ground_truth": "The RAG model improves question answering by combining the retrieval of...",
"contexts": [
"The RAG model integrates document retrieval with LLMs by first retrieving...",
"By incorporating retrieval mechanisms, RAG leverages external knowledge sources...",
]
})
print(f"Context Precision: {eval_result['context_precision']}")